# PawLens — Training Notebook (Google Colab)

Trains all three PawLens models and writes them, with their metrics, to `models/`.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

| Step | Model | Approx. time on a T4 |
|---|---|---|
| 1 | Custom CNN (baseline) | 15–25 min |
| 2 | MobileNetV2 (species) | 15–20 min |
| 3 | EfficientNetB0 (breeds) | 15–25 min |

At the end you download `pawlens_models.zip` and unzip it into `backend/models/` of the project.

## 0. Upload the `training` folder
Upload `training.zip` (zip of the project's `training/` folder) using the file panel on the left, then run:

In [ ]:
!unzip -o -q training.zip -d /content/
%cd /content/training
!pip install -q -r requirements.txt
import os; os.environ["PAWLENS_OUT_DIR"] = "/content/models"
!nvidia-smi -L

## 1. Quick check (1 minute)
Runs every script on a few images to catch errors before the long runs. Optional.

In [ ]:
!python train_custom_cnn.py --limit 64 --epochs 1
!python train_species.py --limit 64 --epochs 1
!python train_breed.py --limit 64 --epochs 1

## 2. Baseline CNN from scratch

In [ ]:
!python train_custom_cnn.py

## 3. MobileNetV2 transfer learning (cat vs dog)

In [ ]:
!python train_species.py

## 4. EfficientNetB0 breed model (37 breeds)

In [ ]:
!python train_breed.py

## 5. Results

In [ ]:
import json, glob
for f in sorted(glob.glob("/content/models/metrics/*.json")):
    m = json.load(open(f))
    top3 = f", top-3 {m['top3_accuracy']:.2%}" if m.get("top3_accuracy") else ""
    print(f"{m['model']:<24} accuracy {m['test_accuracy']:.2%}  F1 {m['f1']:.3f}{top3}  params {m['params']:,}")

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob("/content/models/metrics/plots/*.png")):
    print(p); display(Image(p, width=700))

## 6. Download the models
Unzip this into `backend/models/` so it contains the `.keras` files, `breed_labels.json` and the `metrics/` folder.

In [ ]:
!cd /content/models && zip -q -r /content/pawlens_models.zip .
from google.colab import files
files.download("/content/pawlens_models.zip")